# Capitolo 17 – Algoritmi ricorsivi: somma, Merge Sort, ricerca binaria, sottoarray massimo (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_python.ipynb)

Gli algoritmi con pseudocodice del capitolo, eseguiti sugli esempi delle dispense: la **somma di un array** (iterativa, per dimezzamento e per decremento di uno), **Merge Sort** con `Merge`, la **ricerca binaria ricorsiva** e il problema del **sottoarray massimo** (forza bruta, divide et impera, algoritmo di Kadane). Le stampe riproducono le tracce e i numeri del capitolo (con gli indici da 0).

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/17-algoritmi-ricorsivi/).

Ogni cella di codice si esegue con **Maiusc+Invio**; le celle vanno eseguite in ordine, dall'alto in basso. Python 3, nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `recursive_sum` (snake_case).

## 1. Somma di un array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: un ciclo che accumula le somme parziali. $S(n)=3n+4$, memoria extra $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide et impera sulle due metà (caso base $l=r$). $2n-1$ chiamate, $S(n)=7n-5$, profondità $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decremento di uno: $S(n)=3n-1$, profondità $n$.

*Input*: un array di $n\ge1$ interi (per `RecursiveSum`, due indici $l\le r$). *Output*: la somma degli elementi. I contatori `steps` (righe di pseudocodice eseguite), `calls` e `maxDepth` (chiamate contemporaneamente attive) misurano le grandezze analizzate nel capitolo; in `RecursiveSum` e `RecursiveSumLinear` la `return` è unica per poter aggiornare la profondità.

Sull'array dell'esempio $a=[9,8,12,3,5,14,6]$ ($n=7$) la somma è $57$; `RecursiveSum` fa $13$ chiamate e $44$ passi, con profondità $4$. Poi si esegue la traccia della tabella del capitolo su $a=[9,8,12,3,5]$: la colonna *stack* riporta le chiamate attive.

In [1]:
steps = 0       # passi eseguiti (righe di pseudocodice)
calls = 0       # chiamate eseguite
depth = 0       # chiamate attive in questo momento
max_depth = 0   # profondità massima della ricorsione

def reset():
    global steps, calls, depth, max_depth
    steps = calls = depth = max_depth = 0

def sum_iterative(a, n):
    """Somma iterativa di a[0..n-1]."""
    global steps
    s = 0
    i = 0
    steps += 2                  # righe 1-2: s := 0, i := 1
    while i < n:
        steps += 3              # test vero + due righe del corpo
        s = s + a[i]
        i = i + 1
    steps += 2                  # test falso + return
    return s

def recursive_sum(a, l, r):
    """Somma di a[l..r] (indici da 0), l <= r, per divide et impera."""
    global steps, calls, depth, max_depth
    calls += 1
    depth += 1
    max_depth = max(max_depth, depth)
    steps += 1                  # riga 1: if l = r
    if l == r:
        steps += 1              # riga 2: return a[l] (caso base)
        result = a[l]
    else:
        steps += 4              # righe 3-6: m, due chiamate, return
        m = l + (r - l) // 2    # divide
        s1 = recursive_sum(a, l, m)         # conquer (metà sinistra)
        s2 = recursive_sum(a, m + 1, r)     # conquer (metà destra)
        result = s1 + s2                    # combine
    depth -= 1
    return result

def recursive_sum_linear(a, n):
    """Somma di a[0..n-1] per decremento di uno, n >= 1."""
    global steps, calls, depth, max_depth
    calls += 1
    depth += 1
    max_depth = max(max_depth, depth)
    steps += 1                  # riga 1: if n = 1
    if n == 1:
        steps += 1              # riga 2: return a[1]
        result = a[0]
    else:
        steps += 2              # righe 3-4: chiamata e return
        s = recursive_sum_linear(a, n - 1)
        result = s + a[n - 1]
    depth -= 1
    return result

# ---- esempio del capitolo ----
a = [9, 8, 12, 3, 5, 14, 6]
n = len(a)
print("a =", a, "  n =", n)
reset(); s = sum_iterative(a, n)
print(f"SumIterative       = {s}   steps={steps} (3n+4={3 * n + 4})")
reset(); s = recursive_sum(a, 0, n - 1)
print(f"RecursiveSum       = {s}   calls={calls} (2n-1={2 * n - 1})  steps={steps} (7n-5={7 * n - 5})  maxDepth={max_depth}")
reset(); s = recursive_sum_linear(a, n)
print(f"RecursiveSumLinear = {s}   calls={calls}  steps={steps} (3n-1={3 * n - 1})  maxDepth={max_depth}")

# ---- traccia di RecursiveSum ----
row = 0
max_stack = 0

def emit(stack, action):
    global row, max_stack
    row += 1
    max_stack = max(max_stack, len(stack))
    print(f"{row:>2d} | {','.join(stack)} | {action}")

def recursive_sum_trace(a, l, r, stack):
    stack.append(f"({l},{r})")
    if l == r:
        emit(stack, f"l=r => return a[{l}]={a[l]}")
        stack.pop()
        return a[l]
    m = l + (r - l) // 2
    emit(stack, f"l!=r, m={l}+({r}-{l})/2={m}; call ({l},{m})")
    s1 = recursive_sum_trace(a, l, m, stack)
    emit(stack, f"s1={s1}; call ({m + 1},{r})")
    s2 = recursive_sum_trace(a, m + 1, r, stack)
    emit(stack, f"s2={s2}; return {s1}+{s2}={s1 + s2}")
    stack.pop()
    return s1 + s2

b = [9, 8, 12, 3, 5]
print()
print("a =", b, "  RecursiveSum(a, 0, 4)")
print(" # | stack | azione")
res = recursive_sum_trace(b, 0, len(b) - 1, [])
print(f"risultato {res}, righe {row}, massimo numero di chiamate attive {max_stack}")

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | azione
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Prova tu.** Cambia l'array `a` (anche la lunghezza) e confronta passi, chiamate e profondità: solo `RecursiveSumLinear` ha profondità $n$.

In [2]:
# Cambia l'array: confronta passi, chiamate e profondità dei tre algoritmi.
a = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
n = len(a)
reset(); print(sum_iterative(a, n), steps)
reset(); print(recursive_sum(a, 0, n - 1), steps, calls, max_depth)
reset(); print(recursive_sum_linear(a, n), steps, calls, max_depth)

136 52
136 107 31 5
136 47 16 16


## 2. Merge Sort e Merge

- **MergeSort**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: il sotto-array $a[l..r]$ ordinato in modo non decrescente. Si chiama `MergeSort(a, 1, n)` nello pseudocodice, `merge_sort(a, 0, n-1)` nel codice.
- **Merge**$(a,l,m,r)$ — *Input*: un array e indici $l\le m<r$ tali che $a[l..m]$ e $a[m+1..r]$ siano ordinati. *Output*: $a[l..r]$ ordinato. Copia le due metà in $L$ e $R$ e riversa in $a$, passo dopo passo, il minore fra i primi due elementi non ancora copiati.

Il contatore `comparisons` conta i confronti fra elementi eseguiti da `Merge`. Sull'array dell'esempio $a=[9,12,3,16,19,11,12]$ si stampano le chiamate di `Merge` nell'ordine di esecuzione (con l'array dopo ciascuna, tabella del capitolo; gli indici sono da 0: aggiungendo 1 si ritrovano quelli del capitolo), poi la traccia di `Merge(a, 0, 1, 3)` su $a=[5,8,2,7]$ (esempio dell'implementazione in C++).

In [3]:
comparisons = 0   # confronti fra elementi eseguiti da Merge
ms_calls = 0      # chiamate di MergeSort

def merge(a, l, m, r):
    """Merge(a, l, m, r): fonde a[l..m] e a[m+1..r], entrambi ordinati."""
    global comparisons
    n1 = m - l + 1                  # dimensione della parte sinistra
    n2 = r - m                      # dimensione della parte destra
    L = [0] * n1                    # array ausiliari
    R = [0] * n2
    for i in range(n1):
        L[i] = a[l + i]
    for j in range(n2):
        R[j] = a[m + 1 + j]
    i = 0
    j = 0
    k = l
    while i < n1 and j < n2:        # fonde L e R in a[l..r]
        comparisons += 1
        if L[i] <= R[j]:
            a[k] = L[i]
            i += 1
        else:
            a[k] = R[j]
            j += 1
        k += 1
    while i < n1:                   # copia il resto di L
        a[k] = L[i]
        i += 1
        k += 1
    while j < n2:                   # copia il resto di R
        a[k] = R[j]
        j += 1
        k += 1

def merge_sort(a, l, r):
    """MergeSort(a, l, r): ordina a[l..r]."""
    global ms_calls
    ms_calls += 1
    if l < r:
        m = l + (r - l) // 2        # divide
        merge_sort(a, l, m)         # conquer (metà sinistra)
        merge_sort(a, m + 1, r)     # conquer (metà destra)
        merge(a, l, m, r)           # combine

# ---- varianti che stampano la traccia ----
def merge_sort_log(a, l, r):
    if l < r:
        m = l + (r - l) // 2
        merge_sort_log(a, l, m)
        merge_sort_log(a, m + 1, r)
        left, right = a[l:m + 1], a[m + 1:r + 1]
        merge(a, l, m, r)
        print(f"Merge(a,{l},{m},{r})  {left} e {right}  ->  {a}")

def merge_trace(a, l, m, r):
    n1 = m - l + 1
    n2 = r - m
    L = a[l:m + 1]
    R = a[m + 1:r + 1]
    print(f"Merge(a, {l}, {m}, {r}) su a = {a}")
    print(f"n1={n1}, n2={n2}, L={L}, R={R}")
    i = j = 0
    k = l
    it = 0
    while i < n1 and j < n2:
        it += 1
        head = f"{it:>2d}. ciclo 1: L[{i}]={L[i]} <= R[{j}]={R[j]}? "
        if L[i] <= R[j]:
            head += f"Sì => a[{k}] = L[{i}] = {L[i]}"
            a[k] = L[i]
            i += 1
        else:
            head += f"No => a[{k}] = R[{j}] = {R[j]}"
            a[k] = R[j]
            j += 1
        k += 1
        print(f"{head}, i={i}, j={j}, k={k}; a={a}")
    while i < n1:
        a[k] = L[i]
        print(f"    ciclo 2 (resto di L): a[{k}] = L[{i}] = {L[i]}; a={a}")
        i += 1
        k += 1
    while j < n2:
        a[k] = R[j]
        print(f"    ciclo 3 (resto di R): a[{k}] = R[{j}] = {R[j]}; a={a}")
        j += 1
        k += 1
    print(f"risultato a = {a}, confronti = {it}")

a = [9, 12, 3, 16, 19, 11, 12]
n = len(a)
print("a =", a, "  n =", n)
print("Chiamate di Merge in ordine di esecuzione:")
merge_sort_log(a[:], 0, n - 1)
comparisons = 0
ms_calls = 0
merge_sort(a, 0, n - 1)
print(f"ordinato = {a}   chiamate di MergeSort={ms_calls} (2n-1={2 * n - 1})   confronti={comparisons}")
print()
merge_trace([5, 8, 2, 7], 0, 1, 3)

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Chiamate di Merge in ordine di esecuzione:
Merge(a,0,0,1)  [9] e [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] e [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] e [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] e [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] e [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] e [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
ordinato = [3, 9, 11, 12, 12, 16, 19]   chiamate di MergeSort=13 (2n-1=13)   confronti=14

Merge(a, 0, 1, 3) su a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. ciclo 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. ciclo 1: L[0]=5 <= R[1]=7? Sì => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. ciclo 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    ciclo 2 (resto di L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
risultato a = [2, 5, 7, 8], confronti = 3


**Prova tu.** Ordina un altro array: il numero di confronti di `Merge` è al massimo $n\lceil\log_2 n\rceil$ e le chiamate di `MergeSort` sono sempre $2n-1$.

In [4]:
# Ordina un altro array e conta i confronti (al massimo n*ceil(log2 n)).
b = [38, 27, 43, 3, 9, 82, 10, 5]
comparisons = 0
ms_calls = 0
merge_sort(b, 0, len(b) - 1)
print(b, comparisons, ms_calls)

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


## 3. RecursiveBinarySearch

**RecursiveBinarySearch**$(a,left,right,m)$ — *Input*: un array $a$ di naturali ordinato in modo non decrescente, due indici $left$ e $right$ e un valore $m$. *Output*: un indice $i\in\{left,\dots,right\}$ con $a[i]=m$, oppure $-1$ se $m$ non è in $a[left..right]$. Ogni chiamata fa **al più una** chiamata ricorsiva sulla metà (circa) dell'intervallo: l'albero delle chiamate è un cammino, con al più $\lfloor\log_2 n\rfloor+2$ chiamate. Per cercare in tutto l'array si chiama `RecursiveBinarySearch(a, 0, n-1, m)`.

Sullo stesso array $a=[3,9,11,12,12,16,19]$ del capitolo si cercano $m=12,10,3,19$. Per ogni ricerca la tabella ha una riga per chiamata ($left$, $right$, $mid$, $a[mid]$, azione), come nel capitolo ma con gli indici da 0: aggiungendo 1 si ritrovano quelli del capitolo (per esempio $m=19$ è trovato all'indice 6, cioè alla posizione 7).

In [5]:
rbs_calls = 0   # chiamate eseguite

def recursive_binary_search(a, left, right, m):
    """Indice di m in a[left..right] (array ordinato), oppure -1."""
    global rbs_calls
    rbs_calls += 1
    if left > right:
        return -1                   # intervallo vuoto: non trovato
    mid = left + (right - left) // 2
    if a[mid] == m:
        return mid                  # trovato
    if a[mid] < m:
        return recursive_binary_search(a, mid + 1, right, m)   # metà destra
    return recursive_binary_search(a, left, mid - 1, m)        # metà sinistra

def rbs_trace(a, left, right, m, k):
    """Come sopra, ma stampa una riga per chiamata (k = numero della chiamata)."""
    if left > right:
        print(f"{k:>4d} {left:>4d} {right:>5d} {'-':>3s} {'-':>6s} left > right => non trovato, return -1")
        return -1
    mid = left + (right - left) // 2
    row = f"{k:>4d} {left:>4d} {right:>5d} {mid:>3d} {a[mid]:>6d} "
    if a[mid] == m:
        print(row + f"a[mid] = m => trovato, return {mid}")
        return mid
    if a[mid] < m:
        print(row + f"a[mid] < m => chiamata su left={mid + 1}, right={right}")
        return rbs_trace(a, mid + 1, right, m, k + 1)
    print(row + f"a[mid] > m => chiamata su left={left}, right={mid - 1}")
    return rbs_trace(a, left, mid - 1, m, k + 1)

a = [3, 9, 11, 12, 12, 16, 19]
n = len(a)
print("a =", a, "  n =", n)
for m in [12, 10, 3, 19]:
    print()
    print(f"Ricerca di m = {m}")
    print(f"{'call':>4s} {'left':>4s} {'right':>5s} {'mid':>3s} {'a[mid]':>6s} azione")
    rbs_trace(a, 0, n - 1, m, 1)
print()
for m in [12, 10, 3, 19]:
    rbs_calls = 0
    r = recursive_binary_search(a, 0, n - 1, m)
    print(f"m={m:<3d} -> {r:>2d} ({r + 1 if r >= 0 else -1:>2d} con indici da 1)  chiamate={rbs_calls}")

a = [3, 9, 11, 12, 12, 16, 19]   n = 7

Ricerca di m = 12
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] = m => trovato, return 3

Ricerca di m = 10
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] > m => chiamata su left=0, right=2
   2    0     2   1      9 a[mid] < m => chiamata su left=2, right=2
   3    2     2   2     11 a[mid] > m => chiamata su left=2, right=1
   4    2     1   -      - left > right => non trovato, return -1

Ricerca di m = 3
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] > m => chiamata su left=0, right=2
   2    0     2   1      9 a[mid] > m => chiamata su left=0, right=0
   3    0     0   0      3 a[mid] = m => trovato, return 0

Ricerca di m = 19
call left right mid a[mid] azione
   1    0     6   3     12 a[mid] < m => chiamata su left=4, right=6
   2    4     6   5     16 a[mid] < m => chiamata su left=6, right=6
   3    6     6   6     19 a[mid] = m => trovato, return 6

m=12  ->  3 ( 4 con ind

**Prova tu.** Cerca un altro valore (presente o assente) e verifica che le chiamate siano al più $\lfloor\log_2 n\rfloor+2$.

In [6]:
# Cerca un altro valore, anche assente: le chiamate sono al più floor(log2 n) + 2.
rbs_calls = 0
print(recursive_binary_search(a, 0, len(a) - 1, 16), rbs_calls)
rbs_calls = 0
print(recursive_binary_search(a, 0, len(a) - 1, 100), rbs_calls)

5 2
-1 4


## 4. BruteForceMaximumSubarray

**BruteForceMaximumSubarray**$(a,n)$ — *Input*: un array $a[1..n]$ di $n$ interi. *Output*: gli indici $bl\le br$ di un sottoarray di somma massima e la somma $best$. Esamina tutti gli $n(n+1)/2$ sottoarray $a[l..r]$ (non vuoti, di elementi consecutivi), calcolando le somme in modo incrementale: $\Theta(n^2)$.

Sull'esempio $a=[-2,5,-1,3]$ del capitolo la tabella ha una riga per sottoarray esaminato ($l$, $r$, somma, azione; in ogni riga che non migliora si indica $best$ corrente). Gli indici sono da 0: il capitolo trova $(2,4,7)$, cioè $a[2..4]=[5,-1,3]$ di somma $7$, qui $(1,3,7)$.

In [7]:
def brute_force_maximum_subarray(a, n):
    """Restituisce (bl, br, best): un sottoarray di somma massima di a[0..n-1]."""
    best = a[0]
    bl = 0
    br = 0
    l = 0
    while l < n:
        s = 0
        r = l
        while r < n:
            s = s + a[r]                  # somma di a[l..r]
            if s > best:
                best = s
                bl = l
                br = r
            r = r + 1
        l = l + 1
    return (bl, br, best)

def brute_force_trace(a, n):
    """Come sopra, ma stampa una riga per ogni sottoarray esaminato."""
    best = a[0]
    bl = 0
    br = 0
    for l in range(n):
        s = 0
        for r in range(l, n):
            s = s + a[r]
            if s > best:
                best, bl, br = s, l, r
                action = f"sum > best => best={best}, bl={bl}, br={br}"
            else:
                action = f"sum <= best={best}"
            print(f"{l:>3d} {r:>3d} {s:>5d}  {action}")
    return (bl, br, best)

a = [-2, 5, -1, 3]
n = len(a)
print("a =", a, "  n =", n, "  sottoarray n(n+1)/2 =", n * (n + 1) // 2)
print(f"{'l':>3s} {'r':>3s} {'sum':>5s}  azione")
bl, br, best = brute_force_trace(a, n)
print(f"risultato (bl, br, best) = ({bl}, {br}, {best})   ({bl + 1}, {br + 1}, {best} con indici da 1)")
print("BruteForceMaximumSubarray(a, n) =", brute_force_maximum_subarray(a, n))

a = [-2, 5, -1, 3]   n = 4   sottoarray n(n+1)/2 = 10
  l   r   sum  azione
  0   0    -2  sum <= best=-2
  0   1     3  sum > best => best=3, bl=0, br=1
  0   2     2  sum <= best=3
  0   3     5  sum > best => best=5, bl=0, br=3
  1   1     5  sum <= best=5
  1   2     4  sum <= best=5
  1   3     7  sum > best => best=7, bl=1, br=3
  2   2    -1  sum <= best=7
  2   3     2  sum <= best=7
  3   3     3  sum <= best=7
risultato (bl, br, best) = (1, 3, 7)   (2, 4, 7 con indici da 1)
BruteForceMaximumSubarray(a, n) = (1, 3, 7)


**Prova tu.** Cambia l'array (per esempio con soli numeri negativi): il sottoarray massimo è il singolo elemento maggiore.

In [8]:
# Prova con un array di soli numeri negativi: il sottoarray massimo è il singolo elemento maggiore.
b = [-3, -1, -2, -5]
print(brute_force_maximum_subarray(b, len(b)))

(1, 1, -1)


## 5. FindMaxCrossingSubarray e FindMaximumSubarray (divide et impera)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: un array e tre indici $l\le mid<r$. *Output*: gli indici $maxleft\le mid<maxright$ di un sottoarray di somma massima fra quelli che attraversano il punto medio, e la sua somma. Scandisce la metà sinistra da $mid$ a $l$ e la destra da $mid+1$ a $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: gli indici $i\le j$ di un sottoarray di somma massima di $a[l..r]$ e la sua somma. Il migliore fra il sottoarray massimo della metà sinistra, della destra e di quelli che attraversano il punto medio (a parità, prima il sinistro, poi il destro). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ chiamate.

Il $-\infty$ dello pseudocodice è il valore più piccolo rappresentabile. Prima la traccia sull'esempio $a=[-2,5,-1,3]$ (come l'esecuzione dell'implementazione C++ del capitolo; risultato $(1,3,7)$, cioè $(2,4,7)$ con indici da 1), poi l'esempio della compravendita di azioni, con le variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: tabella dei risultati delle chiamate sui sottointervalli (per livello dell'albero, come nel capitolo; gli indici sono da 0) e i calcoli dei due `FindMaxCrossingSubarray` del livello 1. Il risultato è $(7,10,43)$, cioè $(8,11,43)$ con indici da 1.

In [9]:
fms_calls = 0   # chiamate di FindMaximumSubarray

def find_max_crossing_subarray(a, l, mid, r):
    """Restituisce (maxleft, maxright, somma) del miglior sottoarray che attraversa mid."""
    leftsum = float("-inf")
    s = 0
    maxleft = mid
    for i in range(mid, l - 1, -1):         # da mid a l
        s = s + a[i]                        # somma di a[i..mid]
        if s > leftsum:
            leftsum = s
            maxleft = i
    rightsum = float("-inf")
    s = 0
    maxright = mid + 1
    for j in range(mid + 1, r + 1):         # da mid+1 a r
        s = s + a[j]                        # somma di a[mid+1..j]
        if s > rightsum:
            rightsum = s
            maxright = j
    return (maxleft, maxright, leftsum + rightsum)

def find_maximum_subarray(a, l, r):
    """Restituisce (i, j, somma) di un sottoarray di somma massima di a[l..r]."""
    global fms_calls
    fms_calls += 1
    if l == r:
        return (l, r, a[l])                 # caso base: un elemento
    mid = l + (r - l) // 2
    ll, lr, lsum = find_maximum_subarray(a, l, mid)
    rl, rr, rsum = find_maximum_subarray(a, mid + 1, r)
    cl, cr, csum = find_max_crossing_subarray(a, l, mid, r)
    if lsum >= rsum and lsum >= csum:
        return (ll, lr, lsum)
    if rsum >= lsum and rsum >= csum:
        return (rl, rr, rsum)
    return (cl, cr, csum)

# ---- varianti che stampano la traccia ----
def fms_trace(a, l, r, depth):
    pad = "  " * depth
    if l == r:
        print(f"{pad}FindMaximumSubarray(a, {l}, {r}) = ({l}, {r}, {a[l]})  caso base")
        return (l, r, a[l])
    mid = l + (r - l) // 2
    print(f"{pad}FindMaximumSubarray(a, {l}, {r}): l != r, mid = {mid}")
    left = fms_trace(a, l, mid, depth + 1)
    right = fms_trace(a, mid + 1, r, depth + 1)
    cross = find_max_crossing_subarray(a, l, mid, r)
    print(f"{pad}  FindMaxCrossingSubarray(a, {l}, {mid}, {r}) = {cross}")
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    print(f"{pad}  left.sum={left[2]}, right.sum={right[2]}, cross.sum={cross[2]} => restituisce {ret}")
    return ret

def fms_rows(a, l, r, rows):
    if l == r:
        return (l, r, a[l])
    mid = l + (r - l) // 2
    left = fms_rows(a, l, mid, rows)
    right = fms_rows(a, mid + 1, r, rows)
    cross = find_max_crossing_subarray(a, l, mid, r)
    if left[2] >= right[2] and left[2] >= cross[2]:
        ret = left
    elif right[2] >= left[2] and right[2] >= cross[2]:
        ret = right
    else:
        ret = cross
    rows.append((l, r, mid, left, right, cross, ret))
    return ret

def crossing_trace(a, l, mid, r):
    print(f"FindMaxCrossingSubarray(a, {l}, {mid}, {r})")
    print(f"{'i':>3s} {'a[i]':>5s} {'sum':>5s} {'leftsum':>8s} {'maxleft':>8s}")
    leftsum = float("-inf")
    s = 0
    for i in range(mid, l - 1, -1):
        s = s + a[i]
        if s > leftsum:
            leftsum, maxleft = s, i
        print(f"{i:>3d} {a[i]:>5d} {s:>5d} {leftsum:>8d} {maxleft:>8d}")
    print(f"{'j':>3s} {'a[j]':>5s} {'sum':>5s} {'rightsum':>8s} {'maxright':>8s}")
    rightsum = float("-inf")
    s = 0
    for j in range(mid + 1, r + 1):
        s = s + a[j]
        if s > rightsum:
            rightsum, maxright = s, j
        print(f"{j:>3d} {a[j]:>5d} {s:>5d} {rightsum:>8d} {maxright:>8d}")
    print(f"sottoarray che attraversa a[{maxleft}..{maxright}], somma {leftsum} + {rightsum} = {leftsum + rightsum}")

a = [-2, 5, -1, 3]
print("a =", a)
res = fms_trace(a, 0, len(a) - 1, 0)
fms_calls = 0
find_maximum_subarray(a, 0, len(a) - 1)
print(f"risultato {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} con indici da 1)   chiamate={fms_calls} (2n-1={2 * len(a) - 1})")

print()
a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
n = len(a)
print("a =", a)
fms_calls = 0
res = find_maximum_subarray(a, 0, n - 1)
print(f"FindMaximumSubarray(a, 0, {n - 1}) = {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} con indici da 1)   chiamate={fms_calls} (2n-1={2 * n - 1})")
rows = []
fms_rows(a, 0, n - 1, rows)
rows.sort(key=lambda t: (-(t[1] - t[0]), t[0]))
print(f"{'call':<9s} {'mid':>3s}  {'left':<13s} {'right':<13s} {'crossing':<13s} restituito")
for (l, r, mid, left, right, cross, ret) in rows:
    print(f"{'[' + str(l) + '..' + str(r) + ']':<9s} {mid:>3d}  {str(left):<13s} {str(right):<13s} {str(cross):<13s} {ret}")
print()
crossing_trace(a, 0, 3, 7)
print()
crossing_trace(a, 8, 11, 15)

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  caso base
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  caso base
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => restituisce (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  caso base
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  caso base
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => restituisce (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => restituisce (1, 3, 7)
risultato (1, 3, 7)   (2, 4, 7 con indici da 1)   chiamate=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 con indici da 1)   chiamate=31 (2n-1=31)
call      mid  left 

**Prova tu.** Cambia l'array: il sottoarray massimo può avere un solo elemento o attraversare il punto medio; le chiamate sono sempre $2n-1$.

In [10]:
# Cambia l'array: quanti elementi ha il sottoarray massimo? Le chiamate sono sempre 2n-1.
b = [4, -1, 2, 1, -5, 4, -7, 3, 3]
fms_calls = 0
print(find_maximum_subarray(b, 0, len(b) - 1), fms_calls)

(0, 3, 6) 17


## 6. Kadane

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n$ interi. *Output*: gli indici $bl\le br$ di un sottoarray di somma massima e la somma $best$. Una sola scansione: $cur$ è la somma massima di un sottoarray che **termina** nella posizione corrente e $cl$ il suo primo indice; se $cur<0$ si riparte da $a[i]$, altrimenti si estende. $\Theta(n)$ passi, memoria extra $\Theta(1)$. (Nel capitolo 18 si trova la versione compatta che restituisce solo la somma.)

Sull'esempio della compravendita di azioni (variazioni giornaliere $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$) si stampa la tabella del capitolo: una riga per iterazione, con i valori alla fine dell'iterazione (indici da 0, la riga $i=0$ è l'inizializzazione). Il risultato $(7,10,43)$, cioè $(8,11,43)$ con indici da 1, coincide con quello dell'algoritmo divide et impera.

In [11]:
def kadane(a, n):
    """Restituisce (bl, br, best): un sottoarray di somma massima di a[0..n-1]."""
    best = a[0]
    bl = 0
    br = 0
    cur = a[0]                  # somma massima di un sottoarray che termina in i
    cl = 0
    i = 1
    while i < n:
        if cur < 0:
            cur = a[i]          # riparte
            cl = i
        else:
            cur = cur + a[i]    # estende
        if cur > best:
            best = cur
            bl = cl
            br = i
        i = i + 1
    return (bl, br, best)

def kadane_trace(a, n):
    """Come sopra, ma stampa una riga per iterazione."""
    best, bl, br = a[0], 0, 0
    cur, cl = a[0], 0
    print(f"{0:>3d} {a[0]:>5d}  {'inizializzazione':<24s} {cur:>4d} {cl:>3d} {best:>5d}  ({bl}, {br})")
    for i in range(1, n):
        if cur < 0:
            cur, cl = a[i], i
            action = "riparte (cur<0)"
        else:
            cur = cur + a[i]
            action = "estende"
        if cur > best:
            best, bl, br = cur, cl, i
        print(f"{i:>3d} {a[i]:>5d}  {action:<24s} {cur:>4d} {cl:>3d} {best:>5d}  ({bl}, {br})")
    return (bl, br, best)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
n = len(a)
print("a =", a)
print(f"{'i':>3s} {'a[i]':>5s}  {'azione':<24s} {'cur':>4s} {'cl':>3s} {'best':>5s}  (bl, br)")
res = kadane_trace(a, n)
print(f"risultato {res}   ({res[0] + 1}, {res[1] + 1}, {res[2]} con indici da 1)")
b = [-2, 5, -1, 3]
print("a =", b, " Kadane(a, 4) =", kadane(b, len(b)))

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
  i  a[i]  azione                    cur  cl  best  (bl, br)
  0    13  inizializzazione           13   0    13  (0, 0)
  1    -3  estende                    10   0    13  (0, 0)
  2   -25  estende                   -15   0    13  (0, 0)
  3    20  riparte (cur<0)            20   3    20  (3, 3)
  4    -3  estende                    17   3    20  (3, 3)
  5   -16  estende                     1   3    20  (3, 3)
  6   -23  estende                   -22   3    20  (3, 3)
  7    18  riparte (cur<0)            18   7    20  (3, 3)
  8    20  estende                    38   7    38  (7, 8)
  9    -7  estende                    31   7    38  (7, 8)
 10    12  estende                    43   7    43  (7, 10)
 11    -5  estende                    38   7    43  (7, 10)
 12   -22  estende                    16   7    43  (7, 10)
 13    15  estende                    31   7    43  (7, 10)
 14    -4  estende                   

**Prova tu.** Cambia l'array (per esempio con soli numeri negativi): Kadane riparte a ogni passo e trova il massimo elemento.

In [12]:
# Prova con un array di soli numeri negativi: Kadane riparte a ogni passo e trova il massimo elemento.
c = [-3, -1, -2, -5]
print(kadane(c, len(c)))

(1, 1, -1)
